# Notebook 03 — Feature Engineering

**Phase 3 learning checkpoint.**  We have clean data. Now we *create new columns* that should be more informative to a model than the raw inputs.

Domain knowledge encoded as features tends to beat algorithmic cleverness on raw features. This is where the real estate side of your brain has to talk to the ML side.

## What you will do here

1. Load the cleaned Ames data + Zillow ZHVI.
2. Walk through six families of engineered features, with before/after correlation checks where it matters.
3. See the Zillow market index — our first cross-dataset feature — get joined onto each home.
4. Save the engineered DataFrame to `data/processed/ames_engineered.csv` so Phase 4 can pick up from here.

## Reading order

Pair this notebook with [`docs/05_feature_engineering.md`](../docs/05_feature_engineering.md) — the doc explains the theory and pitfalls; the notebook shows the consequences.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

from src.data_loader import load_ames, load_zillow_zhvi
from src.preprocessing import clean_ames
from src import features as fe
from src import eda

print("Setup complete.")

## 1. Start with the cleaned data

We load the cleaned Ames frame from Phase 2's saved CSV (faster than re-cleaning each time).

In [ ]:
clean_path = PROJECT_ROOT / "data" / "processed" / "ames_clean.csv"
if clean_path.exists():
    df = pd.read_csv(clean_path)
    print(f"Loaded {clean_path.name}: {df.shape}")
else:
    # Fallback: rerun cleaning if the CSV isn't there
    df = clean_ames(load_ames())
    print(f"Re-cleaned: {df.shape}")

df.head(3)

## 2. Aggregation features — collapse the redundancy

Recall from Phase 2 EDA: `garage_cars` and `garage_area` were both ~0.65 correlated with the target. They are saying the same thing. The three square-footage columns (`1st_flr_sf`, `2nd_flr_sf`, `total_bsmt_sf`) are similar.

**Aggregation features** collapse a group of correlated columns into one informative total. Let's see how much that helps.

In [ ]:
# Step 1: correlations of the raw parts with saleprice
parts = ["1st_flr_sf", "2nd_flr_sf", "total_bsmt_sf", "gr_liv_area"]
before = df[parts + ["saleprice"]].corr()["saleprice"].drop("saleprice")
print("BEFORE -- per-part correlations with saleprice:")
print(before.round(3).to_string(), "\n")

# Step 2: engineer total_sf and check
df = fe.add_total_sf(df)
df = fe.add_total_bathrooms(df)
df = fe.add_total_porch_sf(df)

after = df[["total_sf", "total_bath", "total_porch_sf", "gr_liv_area", "saleprice"]].corr()["saleprice"].drop("saleprice")
print("AFTER -- aggregated correlations with saleprice:")
print(after.round(3).to_string())

Notice how `total_sf` correlates *more strongly* than any of its individual parts. That's the signal aggregation captures.

We *kept* the original columns — we don't have to choose. A tree-based model in Phase 4 will pick the most useful split; a linear model may benefit from manual feature selection later.

## 3. Temporal features — age beats year

`year_built = 1995` is a fact. `home_age = 13 years at sale` is a model-friendly framing of the same fact. Let's see the impact.

In [ ]:
before = df[["year_built", "year_remod_add", "saleprice"]].corr()["saleprice"].drop("saleprice")
print("BEFORE -- year columns correlations:")
print(before.round(3).to_string(), "\n")

df = fe.add_age_features(df)

after = df[["home_age", "years_since_remodel", "is_remodeled", "is_new", "saleprice"]].corr()["saleprice"].drop("saleprice")
print("AFTER -- age-derived correlations:")
print(after.round(3).to_string())

Two things:

1. **`home_age` has the *same magnitude* of correlation as `year_built` but negative.** That's expected — older home, lower price. The information is equivalent.
2. **`is_new` is itself a small but real signal.** New construction commands a premium — the model can pick that up directly now.

Trees will use these features as cleaner splits; linear models may prefer the centered, age-based version because it has a more intuitive zero point.

## 4. Cyclic month encoding — the sin/cos trick

`mo_sold = 12` (December) is closer to `mo_sold = 1` (January) than to `mo_sold = 6` (June). The integer encoding hides this — a regression sees December as 11 units further from January than November.

We map each month to a point on a unit circle:

```
       (cos, sin)
      Jan = (1, 0)
      Apr = (0, 1)
      Jul = (-1, 0)
      Oct = (0, -1)
      Dec ≈ (1, -0)   <-- close to Jan!
```

In [ ]:
df = fe.add_cyclic_month(df, col="mo_sold")

# Visualize the cyclic encoding
months = pd.DataFrame({"month": range(1, 13)})
months["sin"] = np.sin(2 * np.pi * months["month"] / 12)
months["cos"] = np.cos(2 * np.pi * months["month"] / 12)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(months["cos"], months["sin"])
for _, row in months.iterrows():
    ax.annotate(int(row["month"]), (row["cos"] + 0.04, row["sin"] + 0.04))
ax.set_xlabel("mo_sold_cos")
ax.set_ylabel("mo_sold_sin")
ax.set_title("Each month is a point on the unit circle")
ax.axhline(0, color="grey", lw=0.5)
ax.axvline(0, color="grey", lw=0.5)
ax.set_aspect("equal")
plt.show()

# How does each form correlate with the target?
print("mo_sold (integer)  corr with saleprice:", round(df["mo_sold"].corr(df["saleprice"]), 4))
print("mo_sold_sin       corr with saleprice:", round(df["mo_sold_sin"].corr(df["saleprice"]), 4))
print("mo_sold_cos       corr with saleprice:", round(df["mo_sold_cos"].corr(df["saleprice"]), 4))

Individual Pearson correlations stay small here because Ames doesn't have a strong seasonal pattern (housing markets do vary by month, but the effect is subtle and Ames is a small dataset). The point is that the *encoding* is now correct — a model that uses these can learn seasonal effects, where it could not before.

## 5. Binary flags — "does it have one at all?"

For features like pools, fireplaces, garages, the *existence* often matters more than the *size*. We add 0/1 flags.

In [ ]:
df = fe.add_binary_flags(df)

flag_cols = ["has_pool", "has_2nd_floor", "has_basement", "has_garage", "has_fireplace", "has_porch"]
summary = pd.DataFrame({
    "share_with": df[flag_cols].mean().round(3),
    "corr_with_saleprice": df[flag_cols + ["saleprice"]].corr()["saleprice"].drop("saleprice").round(3),
})
print("Binary-flag summary:")
summary

`has_fireplace` is a particularly clean signal — about 50% of homes have one, and having one correlates moderately with price. `has_pool` is rare and noisy (very few homes have pools, so the correlation is small).

## 6. Cross-dataset feature — the Ames market index

Ames sales span 2006–2010, straddling the great recession. A $200k home in 2007 was the *same home* listing at $180k in 2010. Without a market-conditions feature, the model has to infer the entire macroeconomic shock from `yr_sold`.

We attach Zillow's monthly ZIP-level home value index (averaged across Ames ZIPs) to each sale.

In [ ]:
zhvi = load_zillow_zhvi()
df = fe.add_market_index(df, zhvi)

print("market index stats:")
print(df["ames_market_index"].describe().round(0).to_string())
print("\ncorr(ames_market_index, saleprice) =", round(df["ames_market_index"].corr(df["saleprice"]), 3))

In [ ]:
# Look at the time-trajectory of the market index that we joined
by_month = (
    df.groupby(["yr_sold", "mo_sold"])["ames_market_index"]
      .first()  # one value per (year, month)
      .reset_index()
)
by_month["date"] = pd.to_datetime(
    by_month["yr_sold"].astype(str) + "-" + by_month["mo_sold"].astype(str).str.zfill(2) + "-15"
)
by_month = by_month.sort_values("date")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_month["date"], by_month["ames_market_index"], marker=".")
ax.set_title("Ames market index (Zillow ZHVI, averaged across Ames ZIPs) over the sale period")
ax.set_ylabel("USD")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

You should see a peak in 2007–2008 and a noticeable dip after — that's the housing crash. Each sale in our dataset now carries information about *when* it happened in market-cycle terms.

> ⚠️ **Note.** This is a single Ames-wide index, not per-home ZIP. The Ames dataset only gives us neighborhood names, not zip codes, so we can't do per-home market lookups. If you had per-row ZIP, this feature would be much more powerful.

## 7. Log-transform the target

Phase 2 EDA showed `saleprice` is right-skewed (skew ≈ 1.7). Training a model on `log1p(saleprice)` typically yields lower error and better-behaved residuals. We add the transformed target as a new column; we *don't* drop the original because we need it for reporting predictions in dollars later.

In [ ]:
df = fe.add_log_target(df, target="saleprice")

print("skew(saleprice)      =", round(df["saleprice"].skew(), 3))
print("skew(log_saleprice)  =", round(df["log_saleprice"].skew(), 3))

eda.plot_distribution(df["saleprice"], log=True)

## 8. The big-picture correlation re-rank

After all our engineering, which features now top the correlation list?

In [ ]:
# Exclude the log target itself or it dominates the list trivially
ranked = eda.top_correlations(df.drop(columns=["log_saleprice"], errors="ignore"),
                              target="saleprice", n=20)
ranked.round(3)

Some of our engineered features now sit at the top. `total_sf` and `home_age` typically slot in among the top 10, and `total_bath` is often in the top 15. The model has cleaner signal to learn from.

## 9. Save engineered dataset

Phase 4 (modeling) will load this CSV instead of re-running the whole pipeline.

In [ ]:
out_path = PROJECT_ROOT / "data" / "processed" / "ames_engineered.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {df.shape[0]:,} rows x {df.shape[1]:,} cols -> {out_path}")

## 10. Wrap-up — what did you learn?

Write your answers down before moving on:

1. **Aggregation payoff.** What was the correlation of `total_sf` with `saleprice` compared to the strongest single part (`gr_liv_area`)? Why is the aggregate often stronger?
2. **Age vs year.** Both `year_built` and `home_age` carry the same information. Why might a *linear* model still prefer `home_age`?
3. **The cyclic trick.** December and January are adjacent in time. Explain in your own words how the `(sin, cos)` encoding makes the model aware of that.
4. **Binary flags.** `has_fireplace` has a higher correlation with `saleprice` than `fireplaces` (the integer count) often does. Why might the existence be more useful than the count for a typical home buyer?
5. **The market index.** Look at the time series chart in section 6. Roughly when does the Ames market index drop, and which sale years are most affected? What does this feature *add* that pure `yr_sold` could not?
6. **Log target.** The skew of `saleprice` dropped from ~1.7 to ~0.1 after log. Why does this matter for the residuals-must-be-normal assumption of linear regression?

When you can answer these, you're ready for **Phase 4: ML Model Training**.